# 52 — Paper table export (main bundle)

Single place to build **paper-ready tables** from **existing** notebook artifacts only (no training, no long evals, no new city-swap runs).

**Benchmark bundle:** standard fairness · city-swap · adversarial pair eval · English transfer — all aligned via `71_unified_models_comparison` outputs where possible.

**Outputs**

| Tables + manifest | `notebooks/results/table_export/` |
| Optional previews | `figures/table_export/` |

Source areas: dataset audit, fairness error analysis, integrated gradients extended, unified comparison, cross-run city-swap, adversarial_model_eval, english_transfer_eval (columns on unified full table), and **T15** from `english_eval/english_benchmark_summary.csv` (from `notebooks/english/english_benchmark_aggregate.ipynb`; not `notebooks/60_label_smoothing.ipynb`).


In [1]:
import json
import re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

_CWD = Path.cwd()
NOTEBOOK_DIR = _CWD if _CWD.name == "notebooks" else (_CWD / "notebooks")
REPO_ROOT = NOTEBOOK_DIR.parent
RESULTS = NOTEBOOK_DIR / "results"
OUT = RESULTS / "table_export"
FIG = REPO_ROOT / "figures" / "table_export"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

MANIFEST = {
    "repo_root": str(REPO_ROOT),
    "output_dir": str(OUT),
    "sources": {},
    "exports": [],
}


def note_source(key: str, path: Path):
    MANIFEST["sources"][key] = str(path.relative_to(REPO_ROOT)) if path.exists() else str(path)


def read_csv_optional(path: Path):
    if not path.exists():
        print("MISSING (skip):", path)
        return None
    return pd.read_csv(path)


def round_num(df: pd.DataFrame, cols, ndigits: int = 3):
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round(ndigits)
    return out


def latex_escape(cell) -> str:
    if cell is None or (isinstance(cell, float) and np.isnan(cell)):
        return "---"
    s = str(cell)
    repl = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "#": r"\#",
        "_": r"\_",
    }
    for a, b in repl.items():
        s = s.replace(a, b)
    return s


def df_to_booktabs_tex(df: pd.DataFrame, label: str = "tab:export") -> str:
    """LaTeX tabular (booktabs) — paste into paper; requires \\usepackage{booktabs}."""
    cols = list(df.columns)
    align = "l" + "r" * max(0, len(cols) - 1)
    lines = [
        "% Generated by 52_table_export.ipynb — requires \\usepackage{booktabs}",
        f"% \\label{{{label}}}",
        r"\begin{tabular}{" + align + "}",
        r"\toprule",
    ]
    head = " & ".join("\\textbf{" + latex_escape(c) + "}" for c in cols) + r" \\"
    lines.append(head)
    lines.append(r"\midrule")
    for _, row in df.iterrows():
        cells = []
        for c in cols:
            v = row[c]
            if isinstance(v, (np.floating, float)) or (isinstance(v, str) and re.match(r"^-?\d+\.?\d*$", str(v).strip())):
                try:
                    fv = float(v)
                    if np.isnan(fv):
                        cells.append("---")
                    else:
                        cells.append(f"{fv:.3f}")
                except (TypeError, ValueError):
                    cells.append(latex_escape(v))
            else:
                cells.append(latex_escape(v))
        lines.append(" & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(lines)


def export_pair(df: pd.DataFrame, stem: str, tex_label: Optional[str] = None):
    csv_path = OUT / f"{stem}.csv"
    tex_path = OUT / f"{stem}.tex"
    df.to_csv(csv_path, index=False)
    label = tex_label or stem.replace("_", ":")
    tex_path.write_text(df_to_booktabs_tex(df, label=label) + "\n", encoding="utf-8")
    MANIFEST["exports"].append({"stem": stem, "csv": csv_path.name, "tex": tex_path.name})
    print("Wrote", csv_path.name, "+", tex_path.name)


In [2]:
# --- Unified comparison: main + challenger compact tables ---
p_full = RESULTS / "unified_comparison" / "01_unified_models_full_table.csv"
p_main = RESULTS / "unified_comparison" / "02_main_track_models.csv"
p_chal = RESULTS / "unified_comparison" / "03_challenger_track_models.csv"

note_source("unified_full", p_full)
note_source("unified_main", p_main)
note_source("unified_challenger", p_chal)

cols = [
    "display_name",
    "model_name",
    "track",
    "family_group",
    "method",
    "accuracy",
    "macro_f1",
    "worst_gap",
    "macro_gap",
    "overall_flip_rate",
    "adversarial_pair_flip_rate",
    "adversarial_mean_delta_prob_true_class",
    "adversarial_n_pairs_evaluated",
    "english_transfer_test_accuracy",
    "english_transfer_test_macro_f1",
    "english_transfer_base_eval_slice_accuracy",
    "english_transfer_base_eval_slice_macro_f1",
]

AXIS_ADV_COLS = ["adversarial_pair_flip_rate", "adversarial_mean_delta_prob_true_class", "adversarial_n_pairs_evaluated"]
AXIS_EN_COLS = [
    "english_transfer_test_accuracy",
    "english_transfer_test_macro_f1",
    "english_transfer_base_eval_slice_accuracy",
    "english_transfer_base_eval_slice_macro_f1",
]

def slim_unified(df: pd.DataFrame) -> pd.DataFrame:
    if df is None:
        return None
    use = [c for c in cols if c in df.columns]
    out = df[use].copy()
    num = [c for c in out.columns if c not in ("display_name", "model_name", "track", "family_group", "method")]
    out = round_num(out, num, 3)
    return out

full = read_csv_optional(p_full)
main = read_csv_optional(p_main)
chal = read_csv_optional(p_chal)

if main is not None:
    m = slim_unified(main)
    export_pair(m, "T01_main_track_unified", "tab:main-track")
else:
    if full is not None:
        m = slim_unified(full[full["track"] == "main"])
        export_pair(m, "T01_main_track_unified", "tab:main-track")
    else:
        print("No main-track unified table available.")

if chal is not None:
    export_pair(slim_unified(chal), "T02_challenger_track_unified", "tab:challenger")
elif full is not None:
    export_pair(slim_unified(full[full["track"] == "challenger"]), "T02_challenger_track_unified", "tab:challenger")

if full is not None:
    export_pair(slim_unified(full), "T03_unified_all_tracks_compact", "tab:unified-all")

# --- Per-axis compact slices (main track) from unified full table ---
if full is not None:
    main_only = full[full["track"] == "main"].copy()
    adv_use = ["display_name", "model_name"] + [c for c in AXIS_ADV_COLS if c in main_only.columns]
    if len(adv_use) > 2:
        t11 = main_only[adv_use].copy()
        num11 = [c for c in t11.columns if c not in ("display_name", "model_name")]
        t11 = round_num(t11, num11, 3)
        export_pair(t11, "T11_adversarial_eval_main_track", "tab:adv-eval-main")
    else:
        print("Skip T11: no adversarial columns on unified full table (re-run 71 after 58).")

    en_use = ["display_name", "model_name"] + [c for c in AXIS_EN_COLS if c in main_only.columns]
    if len(en_use) > 2:
        t12 = main_only[en_use].copy()
        num12 = [c for c in t12.columns if c not in ("display_name", "model_name")]
        t12 = round_num(t12, num12, 3)
        export_pair(t12, "T12_english_transfer_main_track", "tab:english-transfer-main")
    else:
        print("Skip T12: no English transfer columns on unified full table (re-run 71 after 59).")


Wrote T01_main_track_unified.csv + T01_main_track_unified.tex
Wrote T02_challenger_track_unified.csv + T02_challenger_track_unified.tex
Wrote T03_unified_all_tracks_compact.csv + T03_unified_all_tracks_compact.tex
Wrote T11_adversarial_eval_main_track.csv + T11_adversarial_eval_main_track.tex
Wrote T12_english_transfer_main_track.csv + T12_english_transfer_main_track.tex


In [3]:
# --- Fairness summary (pre-aggregated main models) ---
p_f = RESULTS / "fairness_error_analysis" / "01_main_model_fairness_metrics.csv"
note_source("fairness_main", p_f)
ff = read_csv_optional(p_f)
if ff is not None:
    ff = round_num(ff, ["accuracy", "macro_f1", "worst_gap", "macro_gap", "overall_flip_rate"], 3)
    export_pair(ff, "T04_fairness_main_models", "tab:fairness-main")


Wrote T04_fairness_main_models.csv + T04_fairness_main_models.tex


In [4]:
# --- City-swap cross-run robustness ---
p_cs = RESULTS / "cross_run_city_swap" / "01_cross_run_city_swap_master.csv"
note_source("city_swap_master", p_cs)
cs = read_csv_optional(p_cs)
if cs is not None:
    ok = cs[cs["status"] == "ok"].copy()
    keep = [
        "display_name",
        "model_name",
        "source_run",
        "track",
        "accuracy",
        "macro_f1",
        "swap_macro_f1",
        "overall_flip_rate",
        "worst_gap",
        "macro_gap",
        "rank_flip_then_swap_f1",
        "rank_fairest_gap_then_f1",
        "balance_robust_quality",
        "delta_swap_f1_minus_standard",
    ]
    use = [c for c in keep if c in ok.columns]
    rob = ok[use].copy()
    num = [c for c in rob.columns if c not in ("display_name", "model_name", "source_run", "track")]
    rob = round_num(rob, num, 3)
    rob = rob.sort_values(["overall_flip_rate", "swap_macro_f1"], ascending=[True, False], na_position="last")
    export_pair(rob, "T05_city_swap_robustness_ok", "tab:city-swap")

    lb = read_csv_optional(RESULTS / "cross_run_city_swap" / "08_city_swap_leaderboard_compact.csv")
    if lb is not None:
        lb.to_csv(OUT / "T05b_city_swap_leaderboard.csv", index=False)
        MANIFEST["exports"].append({"stem": "T05b_city_swap_leaderboard", "csv": "T05b_city_swap_leaderboard.csv", "tex": None})
        print("Wrote T05b_city_swap_leaderboard.csv (CSV only — wide leaderboard)")

    bad = cs[cs["status"] != "ok"][["model_name", "source_run", "status", "error"]]
    if len(bad):
        bad.to_csv(OUT / "T05c_city_swap_failed_rows.csv", index=False)
        MANIFEST["exports"].append({"stem": "T05c_city_swap_failed_rows", "csv": "T05c_city_swap_failed_rows.csv", "tex": None})
        print("Wrote T05c_city_swap_failed_rows.csv")


Wrote T05_city_swap_robustness_ok.csv + T05_city_swap_robustness_ok.tex
Wrote T05b_city_swap_leaderboard.csv (CSV only — wide leaderboard)
Wrote T05c_city_swap_failed_rows.csv


In [5]:
# --- Dataset audit: splits + geography ---
p_split = RESULTS / "dataset_audit" / "02_split_sizes.csv"
p_city = RESULTS / "dataset_audit" / "05_top_city_groups.csv"
p_sc = RESULTS / "dataset_audit" / "03_supercategory_counts_by_split.csv"
note_source("audit_split_sizes", p_split)
note_source("audit_city_groups", p_city)
note_source("audit_supercategory", p_sc)

splits = read_csv_optional(p_split)
if splits is not None:
    export_pair(splits, "T06_dataset_split_sizes", "tab:dataset-splits")

cities = read_csv_optional(p_city)
if cities is not None:
    top = cities.head(12).copy()
    export_pair(top, "T07_dataset_top_city_groups", "tab:dataset-cities")

sc = read_csv_optional(p_sc)
if sc is not None:
    tr = sc.loc[sc["split"] == "train"].drop(columns=["split"])
    if len(tr):
        row = tr.iloc[0].astype(float)
        tdf = row.to_frame(name="train_n").reset_index().rename(columns={"index": "supercategory"})
        tdf = tdf.sort_values("train_n", ascending=False)
        export_pair(tdf, "T08_dataset_train_supercategory_counts", "tab:dataset-supercat")


Wrote T06_dataset_split_sizes.csv + T06_dataset_split_sizes.tex
Wrote T07_dataset_top_city_groups.csv + T07_dataset_top_city_groups.tex
Wrote T08_dataset_train_supercategory_counts.csv + T08_dataset_train_supercategory_counts.tex


In [6]:
# --- Integrated gradients — city attribution mass ---
p_ig = RESULTS / "integrated_gradients_extended" / "07_city_attr_mass_ratio_by_model.csv"
note_source("ig_city_attr", p_ig)
ig = read_csv_optional(p_ig)
if ig is not None:
    ig = round_num(ig, ["mean", "median", "max"], 4)
    export_pair(ig, "T09_ig_city_attr_mass_by_model", "tab:ig-city-mass")


Wrote T09_ig_city_attr_mass_by_model.csv + T09_ig_city_attr_mass_by_model.tex


In [7]:
# --- Merged main-track wide table (standard fairness + city-swap + optional adversarial / English via unified full) ---
main_csv = OUT / "T01_main_track_unified.csv"
fair_csv = OUT / "T04_fairness_main_models.csv"
cs_csv = OUT / "T05_city_swap_robustness_ok.csv"
p_full_unified = RESULTS / "unified_comparison" / "01_unified_models_full_table.csv"

if main_csv.exists() and fair_csv.exists() and cs_csv.exists():
    m = pd.read_csv(main_csv)
    f = pd.read_csv(fair_csv).drop_duplicates(subset=["model_name"])
    c = pd.read_csv(cs_csv)
    # Prefer 70_batch city-swap row per model if duplicates
    if "source_run" in c.columns:
        c["_prio"] = (c["source_run"] == "70_batch").astype(int)
        c = c.sort_values("_prio", ascending=False).drop_duplicates(subset=["model_name"], keep="first").drop(columns=["_prio"])

    fair_flip = f[["model_name", "overall_flip_rate"]].rename(
        columns={"overall_flip_rate": "flip_rate_fairness_csv"}
    )
    merged = m.merge(fair_flip, on="model_name", how="left")
    c_cols = [x for x in ["model_name", "swap_macro_f1", "overall_flip_rate", "balance_robust_quality"] if x in c.columns]
    c_small = c[c_cols].rename(columns={"overall_flip_rate": "flip_rate_cityswap_eval"})
    merged = merged.merge(c_small, on="model_name", how="left")
    merged = merged.rename(columns={"overall_flip_rate": "flip_rate_unified_table"})
    num_cols = [c for c in merged.columns if c not in ("display_name", "model_name", "track", "family_group", "method")]
    merged = round_num(merged, num_cols, 3)
    export_pair(merged, "T10_main_track_merged_standard_fairness_cityswap", "tab:main-merged")

    ufull = read_csv_optional(p_full_unified)
    if ufull is not None:
        extra_cols = [c for c in AXIS_ADV_COLS + AXIS_EN_COLS if c in ufull.columns]
        if extra_cols:
            bundle = merged.merge(ufull[["model_name"] + extra_cols].drop_duplicates(subset=["model_name"]), on="model_name", how="left")
            bnum = [c for c in bundle.columns if c not in ("display_name", "model_name", "track", "family_group", "method")]
            bundle = round_num(bundle, bnum, 3)
            export_pair(bundle, "T14_main_track_merged_benchmark_bundle", "tab:main-benchmark-bundle")
        else:
            print("Skip T14: unified full table has no adversarial/English columns (re-run 71).")
else:
    print("Skip T10/T14 merge: need T01, T04, T05 from this run.")


Wrote T10_main_track_merged_standard_fairness_cityswap.csv + T10_main_track_merged_standard_fairness_cityswap.tex
Wrote T14_main_track_merged_benchmark_bundle.csv + T14_main_track_merged_benchmark_bundle.tex


In [8]:
# --- English benchmark wide table (english_benchmark_aggregate: eval + transfer bundles) ---
p_en_summary = RESULTS / "english_eval" / "english_benchmark_summary.csv"
p_en_legacy = RESULTS / "english_eval" / "60_english_benchmark_summary.csv"
p_en = p_en_summary if p_en_summary.exists() else p_en_legacy
note_source("english_benchmark_summary", p_en)
en60 = read_csv_optional(p_en)
if en60 is not None:
    skip = {"model_run_id", "eval_bundle", "metrics_json", "display_name_ru"}
    num_cols = [c for c in en60.columns if c not in skip]
    en60_slim = round_num(en60, num_cols, 3)
    export_pair(en60_slim, "T15_english_benchmark_wide", "tab:english-benchmark-wide")
else:
    print(
        "Skip T15: run notebooks/english/english_benchmark_aggregate.ipynb "
        "to produce notebooks/results/english_eval/english_benchmark_summary.csv "
        "(legacy filename 60_english_benchmark_summary.csv is still detected if present)."
    )


Wrote T15_english_benchmark_wide.csv + T15_english_benchmark_wide.tex


In [9]:
# --- Optional preview figure (main unified table, first rows) ---
try:
    import matplotlib.pyplot as plt

    p = OUT / "T01_main_track_unified.csv"
    if p.exists():
        prev = pd.read_csv(p).head(10)
        fig, ax = plt.subplots(figsize=(11, 0.35 * (len(prev) + 2)))
        ax.axis("off")
        tbl = ax.table(
            cellText=prev.values,
            colLabels=prev.columns,
            loc="center",
            cellLoc="center",
        )
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(8)
        tbl.scale(1, 1.35)
        fig.suptitle("Preview: T01 main track (first 10 rows)", fontsize=11, y=0.98)
        fig.tight_layout()
        fig.savefig(FIG / "preview_T01_main_track.png", dpi=200, bbox_inches="tight")
        plt.close(fig)
        MANIFEST["figures"] = ["preview_T01_main_track.png"]
        print("Saved figures/table_export/preview_T01_main_track.png")
except Exception as e:
    print("Preview figure skipped:", e)


Saved figures/table_export/preview_T01_main_track.png


In [10]:
# --- Manifest ---
MANIFEST["exports"] = sorted(MANIFEST["exports"], key=lambda x: x["stem"])
(OUT / "manifest.json").write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False), encoding="utf-8")
print("Wrote manifest.json")
list(OUT.glob("*"))


Wrote manifest.json


[PosixPath('/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/table_export/T02_challenger_track_unified.tex'),
 PosixPath('/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/table_export/T11_adversarial_eval_main_track.tex'),
 PosixPath('/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/table_export/T06_dataset_split_sizes.csv'),
 PosixPath('/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/table_export/T05_city_swap_robustness_ok.tex'),
 PosixPath('/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/table_export/T14_main_track_merged_benchmark_bundle.tex'),
 PosixPath('/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/table_export/T08_dataset_train_supercategory_counts.tex'),
 PosixPath('/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/table_expo

## Key takeaways — what to cite where

| Export | Role | Built from |
|--------|------|------------|
| **T01** `main_track_unified` | **Main paper** — primary main-track metrics (accuracy, F1, gaps, flip-rate when present, plus adversarial/English columns when notebook **71** merged them). | `unified_comparison/02_main_track_models.csv` (else filtered `01_unified_models_full_table.csv`) |
| **T02** `challenger_track_unified` | **Main paper / methods** — challenger sweep in the same schema as T01. | `unified_comparison/03_challenger_track_models.csv` |
| **T04** `fairness_main_models` | **Main paper** — fairness table used in error-analysis narrative (`overall_flip_rate` when present). | `fairness_error_analysis/01_main_model_fairness_metrics.csv` |
| **T05** `city_swap_robustness_ok` | **Main paper / robustness** — ok rows only: swap F1, flip, ranks, balance. | `cross_run_city_swap/01_cross_run_city_swap_master.csv` |
| **T10** `main_track_merged_standard_fairness_cityswap` | **Main paper (wide row)** — one row per main model: flip columns from unified vs fairness snapshot vs city-swap merge (70_batch row preferred per model). | Derived here from T01 + T04 + T05 |
| **T11** `adversarial_eval_main_track` | **Robustness axis** — adversarial pair metrics merged in 71 (NaN where 58 not run). | `unified_comparison/01_unified_models_full_table.csv` (main rows) |
| **T12** `english_transfer_main_track` | **Transfer axis** — English test (+ optional base eval slice) from 71 merge. | `unified_comparison/01_unified_models_full_table.csv` (main rows) |
| **T14** `main_track_merged_benchmark_bundle` | **Main paper (wide row)** — T10-style merge **plus** adversarial + English columns from unified full when present. | Derived from T10 merge + `01_unified_models_full_table.csv` |
| **T15** `english_benchmark_wide` | **English / transfer axis** — full wide table from **english_benchmark_aggregate**: per-slice accuracy/F1 + optional Russian merge + Δ(EN−RU) macro-F1. | `english_eval/english_benchmark_summary.csv` (from `notebooks/english/english_benchmark_aggregate.ipynb`) |
| **T03** `unified_all_tracks_compact` | **Appendix** — full unified table in one file. | `unified_comparison/01_unified_models_full_table.csv` |
| **T05b** / **T05c** | **Appendix / debugging** — full leaderboard CSV; failed city-swap loads. | `08_city_swap_leaderboard_compact.csv`, master `status != ok` |
| **T06–T08** | **Data / appendix** — split sizes, top cities, train supercategory counts. | `dataset_audit/02_…`, `05_…`, `03_…` |
| **T09** | **Attribution / appendix** — IG city attribution mass by model shorthand. | `integrated_gradients_extended/07_city_attr_mass_ratio_by_model.csv` |

All `.tex` files use **booktabs** (`\toprule`, `\midrule`, `\bottomrule`); add `\usepackage{booktabs}` in the thesis preamble. Regenerate after upstream CSVs change.
